# 윈도우 라벨 매핑

CloudTrail 이벤트를 **5개씩 윈도우로 분할**할 때, 각 윈도우에 전술(tactic)·기법(technique) 라벨을 어떻게 붙일지 정리한다.

- 이벤트 단위 라벨: 각 이벤트는 `benign`이거나 하나의 (전술, 기법) 라벨을 가진다.
- 윈도우 단위 라벨: 윈도우 안의 5개 이벤트 라벨을 모아 윈도우 라벨을 만든다.

## 방식 1. 공격 이벤트가 하나라도 있으면 공격 라벨

윈도우 안에 **공격 이벤트가 하나라도 있으면** 그 윈도우 전체를 공격으로 라벨링한다.

- 공격 이벤트가 없으면 → `benign`
- 공격 이벤트가 1개 이상이면 → 그 이벤트의 전술·기법 라벨을 윈도우 라벨로 사용
- 서로 다른 기법이 여러 개 섞여 있으면 → 등장한 기법(과 전술)을 모두 포함하는 **멀티라벨**

예시 (윈도우 크기 5):

| 윈도우 | 이벤트 라벨 | 윈도우 라벨 |
|---|---|---|
| w0 | benign ×5 | benign |
| w1 | benign ×4, T1078 | TA0001 / T1078 |
| w2 | T1078, benign ×3, T1530 | {TA0001, TA0009} / {T1078, T1530} |

**장점**
- 공격을 놓치지 않는다 (recall 관점에서 유리).
- 규칙이 단순해서 구현과 설명이 쉽다.

**단점**
- 공격 이벤트가 1개뿐이고 나머지 4개가 정상이어도 공격으로 라벨링되어, 윈도우 텍스트 대부분이 정상 행위인 노이즈 라벨이 생긴다.
- 슬라이딩 윈도우(stride < 5)를 쓰면 같은 공격 이벤트가 여러 윈도우에 공격 라벨을 퍼뜨려 공격 윈도우 수가 부풀려진다.

## 예시 코드

In [ ]:
import pandas as pd

# 이벤트 단위 라벨 예시 (benign이면 tactic/technique/attack_id = None)
# attack_id: 한 번의 기법 실행을 이루는 이벤트 묶음 ID (방식 2에서 사용)
events = pd.DataFrame({
    "eventName": [
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ListUsers",
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ConsoleLogin",
        "ConsoleLogin", "ListUsers", "DescribeRegions", "ListBuckets", "GetObject",
    ],
    "tactic":    [None]*9 + ["TA0001", "TA0001", None, None, None, "TA0009"],
    "technique": [None]*9 + ["T1078",  "T1078",  None, None, None, "T1530"],
    "attack_id": [None]*9 + ["a1",     "a1",     None, None, None, "a2"],
})
events

In [ ]:
def make_windows(df, size=5, stride=5):
    """이벤트를 size개씩 묶는다. stride < size이면 슬라이딩 윈도우."""
    return [df.iloc[i:i + size] for i in range(0, len(df) - size + 1, stride)]


def label_any_attack(window):
    """방식 1: 공격 이벤트가 하나라도 있으면 등장한 전술·기법을 모두 윈도우 라벨로."""
    attack = window.dropna(subset=["technique"])
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

In [ ]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=5)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
pd.DataFrame(rows)

슬라이딩 윈도우(stride=1)에서는 같은 공격 이벤트가 여러 윈도우에 걸쳐 공격 라벨을 만든다.

In [ ]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=1)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
sliding = pd.DataFrame(rows)
print("공격 윈도우 비율:", sliding["is_attack"].mean().round(2))
sliding

## 방식 2. 기법 이벤트 전체가 윈도우 안에 있어야 공격 라벨

하나의 기법 실행(`attack_id`)을 이루는 공격 이벤트가 **잘리지 않고 모두 윈도우 안에 들어 있을 때만** 그 전술·기법을 윈도우 라벨로 붙인다.

- 윈도우 안에 완전히 포함된 기법이 **하나라도 있으면** → 공격 (포함된 기법들의 전술·기법이 라벨, 여러 개면 멀티라벨)
- 완전히 포함된 기법이 **하나도 없으면** → `benign`
  - 공격 이벤트가 윈도우 경계에 걸쳐 일부만 들어 있는 경우도 `benign`으로 본다.

예시 (윈도우 크기 5, 기법 a1 = T1078 이벤트 2개가 w1·w2 경계에 걸침, a2 = T1530 이벤트 1개):

| 윈도우 | 이벤트 라벨 | 방식 1 | 방식 2 |
|---|---|---|---|
| w0 | benign ×5 | benign | benign |
| w1 | benign ×4, a1 일부 | TA0001 / T1078 | benign (a1이 잘림) |
| w2 | a1 일부, benign ×3, a2 | {TA0001, TA0009} / {T1078, T1530} | TA0009 / T1530 |

**장점**
- 라벨이 붙은 윈도우에는 그 기법의 행위가 온전히 들어 있어, 윈도우 텍스트와 라벨이 잘 맞는다 (라벨 정밀도 높음).
- 슬라이딩 윈도우에서도 기법 하나가 공격 라벨을 퍼뜨리는 범위가 줄어든다.

**단점**
- 이벤트 수가 5개보다 많은 기법은 어떤 윈도우에도 완전히 들어갈 수 없어 **영원히 공격으로 라벨링되지 않는다.** 윈도우 크기를 정하기 전에 기법별 이벤트 수 분포를 확인해야 한다.
- 경계에 걸친 공격 이벤트가 있는 윈도우가 `benign`으로 들어가, 정상 데이터에 공격 행위가 섞인다.
- 기법 실행 단위(`attack_id`)를 이벤트에 미리 붙여 두어야 한다.

In [ ]:
def label_fully_contained(window, all_events):
    """방식 2: 이벤트 전체가 윈도우 안에 들어온 기법만 윈도우 라벨로."""
    total = all_events["attack_id"].value_counts()   # 기법 실행별 전체 이벤트 수
    in_win = window["attack_id"].value_counts()      # 윈도우 안에 들어온 이벤트 수
    full_ids = [a for a, n in in_win.items() if n == total[a]]
    attack = window[window["attack_id"].isin(full_ids)]
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

방식 1과 방식 2를 같은 윈도우에서 비교한다.

In [ ]:
def compare(df, size=5, stride=5):
    rows = []
    for i, w in enumerate(make_windows(df, size=size, stride=stride)):
        m1, m2 = label_any_attack(w), label_fully_contained(w, df)
        rows.append({
            "window": f"w{i}",
            "m1_techniques": m1["techniques"] or "benign",
            "m2_techniques": m2["techniques"] or "benign",
        })
    return pd.DataFrame(rows)

compare(events, stride=5)

In [ ]:
# 슬라이딩 윈도우(stride=1)
cmp = compare(events, stride=1)
print("공격 윈도우 수 — 방식 1:", (cmp["m1_techniques"] != "benign").sum(),
      "/ 방식 2:", (cmp["m2_techniques"] != "benign").sum())
cmp

In [ ]:
# 기법 실행별 이벤트 수: 윈도우 크기(5)보다 큰 기법은 방식 2에서 라벨이 붙지 않는다
sizes = events.dropna(subset=["attack_id"]).groupby(["attack_id", "technique"]).size().rename("n_events")
sizes.to_frame().assign(fits_in_window=sizes <= 5)